In [1]:
from pyspark.sql import SparkSession
# 显式导入需要的函数（避免通配符导致命名冲突）
from pyspark.sql.functions import col, lit, date_format, count, sum, when, concat, substring, year, month, current_timestamp
# 显式导入需要的类型（避免*导入的命名污染）
from pyspark.sql.types import StringType, DoubleType
from datetime import datetime
import uuid
import os

# ===================== 1. 定义日期参数 =====================
# 方式1：默认使用当前日期（生产环境用）
current_date_str = datetime.now().strftime("%Y-%m-%d")

# 方式2：测试用固定日期（取消注释即可切换）
# current_date_str = datetime(2025, 12, 1).strftime("%Y-%m-%d")  

# 校验日期格式
try:
    datetime.strptime(current_date_str, "%Y-%m-%d")
except ValueError:
    raise ValueError(f"日期格式错误：{current_date_str}，请使用YYYY-MM-DD格式")

# ===================== 2. 封装SQL查询为方法 =====================
def get_sql_part1(run_date):
    """生成返途中车辆查询SQL，接收日期参数"""
    sql = f"""
SELECT 
    CASE 
        WHEN j8.factory = '6011' AND j10.xh = '2' THEN 'FNR'
        WHEN j8.factory = '6041' AND j10.xh = '2' THEN 'NMR'
        ELSE comp.pyjiancheng  
    END as factoryCode,

    CASE 
        WHEN j8.factory= '6011' AND j10.xh = '2' THEN '6071'
        WHEN j8.factory = '6041' AND j10.xh = '2' THEN '60412'
        ELSE j8.factory 
    END as factory_No,

    CONCAT('20', j8.season) as season,
    '1' as inouttype,
    '返途中to_factory' as inouttypename,
    z31.fl as cxdm,
    z31.cmc as cxmc,
    z32.cllxdm as vehicle_type_code,
    z32.cllx as vehicle_type,
    '{run_date}' as dateTarget,
    COUNT(*) as vehicle_num,
    CASE z31.fl 
        WHEN '1' THEN 15.0
        WHEN '2' THEN 10.0
        WHEN '4' THEN 7.0 
    END as tonnage,
    SUM(CASE z31.fl 
        WHEN '1' THEN 15.0
        WHEN '2' THEN 10.0
        WHEN '4' THEN 7.0 
     END) as Estimated_weight,
    CURRENT_TIMESTAMP() as create_time
FROM dwd.dwd_gis_progress_report_f_1h j8
INNER JOIN dwd.dwd_gis_dispatch_table_f_1h j10 ON j8.zzhn = j10.zzhn 
INNER JOIN dwd.dwd_gis_fleet_info_i_1mon z32 ON j10.cph = z32.cph 
    AND j10.factory = z32.factory 
    AND j10.season = z32.season
INNER JOIN dwd.dwd_gis_vehicle_model_info_i_1mon z31 ON z32.cx = z31.fl 
    AND z32.factory = z31.factory
LEFT JOIN dwd.dwd_gis_fn_nw_company_info_i_1mon comp ON comp.jiagonggc = j8.factory
WHERE COALESCE(j8.zhuangchezt, '0') = '3'
  AND j10.zt = '0'
  AND j10.season = (
    CASE 
      WHEN MONTH('{run_date}') BETWEEN 1 AND 10 THEN 
        CONCAT(SUBSTRING(CAST(YEAR('{run_date}') - 1 AS STRING), 3, 2), '/', SUBSTRING(CAST(YEAR('{run_date}') AS STRING), 3, 2))
      WHEN MONTH('{run_date}') BETWEEN 11 AND 12 THEN 
        CONCAT(SUBSTRING(CAST(YEAR('{run_date}') AS STRING), 3, 2), '/', SUBSTRING(CAST(YEAR('{run_date}') + 1 AS STRING), 3, 2))
    END
  )
  AND j8.zzh NOT IN (
    SELECT zzh 
    FROM dwd.dwd_gis_parking_lot_queue_i_1h w4 
    WHERE w4.factory = j8.factory 
    AND w4.season = j8.season
  )
  AND comp.dt != ''
GROUP BY 
    CASE 
        WHEN j8.factory = '6011' AND j10.xh = '2' THEN 'FNR'
        WHEN j8.factory = '6041' AND j10.xh = '2' THEN 'NMR'
        ELSE comp.pyjiancheng 
    END,
    CASE 
        WHEN j8.factory= '6011' AND j10.xh = '2' THEN '6071'
        WHEN j8.factory = '6041' AND j10.xh = '2' THEN '60412'
        ELSE j8.factory 
    END,
    CONCAT('20', j8.season),
    z31.fl, 
    z31.cmc, 
    z32.cllxdm, 
    z32.cllx;
    """
    return sql

def get_sql_part2(run_date):
    """生成厂外排队车辆查询SQL，接收日期参数"""
    sql = f"""
SELECT 
    CASE 
        WHEN w4.factory = '6011' AND j10.xh = '2' THEN 'FNR'
        WHEN w4.factory = '6041' AND j10.xh = '2' THEN 'NMR'
        ELSE comp.pyjiancheng
    END as factoryCode,
    CASE 
        WHEN w4.factory= '6011' AND j10.xh = '2' THEN '6071'
        WHEN w4.factory = '6041' AND j10.xh = '2' THEN '60412'
        ELSE w4.factory 
    END as factory_No,
    CONCAT('20', w4.season) as season,
    '2' as inouttype,
    '厂外out' as inouttypename,
    z31.fl as cxdm,
    z31.cmc as cxmc,
    z32.cllxdm as vehicle_type_code,
    z32.cllx as vehicle_type,
    '{run_date}' as dateTarget,
    COUNT(*) as vehicle_num,
    CASE z31.fl 
        WHEN '1' THEN 15.0
        WHEN '2' THEN 10.0
        WHEN '4' THEN 7.0 
    END as tonnage,
    -- 优化：用聚合函数写法更规范（等价原逻辑）
    SUM(CASE z31.fl 
        WHEN '1' THEN 15.0
        WHEN '2' THEN 10.0
        WHEN '4' THEN 7.0 
     END) as Estimated_weight,
    CURRENT_TIMESTAMP() as create_time
FROM dwd.dwd_gis_parking_lot_queue_i_1h w4
INNER JOIN dwd.dwd_gis_dispatch_table_f_1h j10 ON w4.zzhn = j10.zzhn 
INNER JOIN dwd.dwd_gis_fleet_info_i_1mon z32 ON j10.cph = z32.cph 
    AND j10.factory = z32.factory 
    AND j10.season = z32.season
INNER JOIN dwd.dwd_gis_vehicle_model_info_i_1mon z31 ON z32.cx = z31.fl 
    AND z32.factory = z31.factory
LEFT JOIN dwd.dwd_gis_fn_nw_company_info_i_1mon comp ON comp.jiagonggc = w4.factory
WHERE w4.season = (
    CASE 
      WHEN MONTH('{run_date}') BETWEEN 1 AND 10 THEN 
        CONCAT(SUBSTRING(CAST(YEAR('{run_date}') - 1 AS STRING), 3, 2), '/', SUBSTRING(CAST(YEAR('{run_date}') AS STRING), 3, 2))
      WHEN MONTH('{run_date}') BETWEEN 11 AND 12 THEN 
        CONCAT(SUBSTRING(CAST(YEAR('{run_date}') AS STRING), 3, 2), '/', SUBSTRING(CAST(YEAR('{run_date}') + 1 AS STRING), 3, 2))
    END
  )
  AND COALESCE(w4.zt, '0') = '0' 
  AND j10.zt = '0'
  AND comp.dt != ''
GROUP BY 
    CASE 
        WHEN w4.factory = '6011' AND j10.xh = '2' THEN 'FNR'
        WHEN w4.factory = '6041' AND j10.xh = '2' THEN 'NMR'
        ELSE comp.pyjiancheng 
    END,
    CASE 
        WHEN w4.factory= '6011' AND j10.xh = '2' THEN '6071'
        WHEN w4.factory = '6041' AND j10.xh = '2' THEN '60412'
        ELSE w4.factory 
    END,
    CONCAT('20', w4.season),
    z31.fl, 
    z31.cmc, 
    z32.cllxdm, 
    z32.cllx;
    """
    return sql

def get_sql_part3(run_date):
    """生成厂内车辆查询SQL，接收日期参数（最终修正：使用g1.cph关联z32）"""
    sql = f"""
SELECT 
    CASE 
        WHEN g1.factory = '6011' AND g1.xh = '2' THEN 'FNR'
        WHEN g1.factory = '6041' AND g1.xh = '2' THEN 'NMR'
        ELSE comp.pyjiancheng
    END as factoryCode,

    CASE 
        WHEN g1.factory = '6011' AND g1.xh = '2' THEN '6071'
        WHEN g1.factory = '6041' AND g1.xh = '2' THEN '60412'
        ELSE g1.factory 
    END as factory_No,

    CONCAT('20', g1.season) as season,
    '3' as inouttype,
    '厂内in' as inouttypename,
    z31.fl as cxdm,
    z31.cmc as cxmc,
    z32.cllxdm as vehicle_type_code,
    z32.cllx as vehicle_type,
    '{run_date}' as dateTarget,
    COUNT(*) as vehicle_num,
    CASE z31.fl 
        WHEN '1' THEN 15.0
        WHEN '2' THEN 10.0
        WHEN '4' THEN 7.0 
    END as tonnage,
    SUM(g1.mz * 0.001 - z32.zcpz) as Estimated_weight,
    CURRENT_TIMESTAMP() as create_time
FROM dwd.dwd_gis_gross_weight_f_1h g1
INNER JOIN dwd.dwd_gis_dispatch_plan_info_i_1h j6 ON g1.zzhn = j6.zzhn 
INNER JOIN dwd.dwd_gis_fleet_info_i_1mon z32 ON g1.cph = z32.cph 
    AND g1.factory = z32.factory 
    AND g1.season = z32.season    
INNER JOIN dwd.dwd_gis_vehicle_model_info_i_1mon z31 ON z32.cx = z31.fl 
    AND z32.factory = z31.factory
LEFT JOIN dwd.dwd_gis_fn_nw_company_info_i_1mon comp ON comp.jiagonggc = g1.factory
WHERE g1.season = (
    CASE 
      WHEN MONTH('{run_date}') BETWEEN 1 AND 10 THEN 
        CONCAT(SUBSTRING(CAST(YEAR('{run_date}') - 1 AS STRING), 3, 2), '/', SUBSTRING(CAST(YEAR('{run_date}') AS STRING), 3, 2))
      WHEN MONTH('{run_date}') BETWEEN 11 AND 12 THEN 
        CONCAT(SUBSTRING(CAST(YEAR('{run_date}') AS STRING), 3, 2), '/', SUBSTRING(CAST(YEAR('{run_date}') + 1 AS STRING), 3, 2))
    END
  )
  AND comp.dt != ''
GROUP BY 
    CASE 
        WHEN g1.factory = '6011' AND g1.xh = '2' THEN 'FNR'
        WHEN g1.factory = '6041' AND g1.xh = '2' THEN 'NMR'
        ELSE comp.pyjiancheng 
    END,
    CASE 
        WHEN g1.factory = '6011' AND g1.xh = '2' THEN '6071'
        WHEN g1.factory = '6041' AND g1.xh = '2' THEN '60412'
        ELSE g1.factory 
    END,
    CONCAT('20', g1.season),
    z31.fl, 
    z31.cmc, 
    z32.cllxdm, 
    z32.cllx;
    """
    return sql

# ===================== 3. 通用工具方法 =====================
def table_exists(spark, table_name):
    """检查表是否存在的替代方法（保留必要try，无其他方案）"""
    try:
        spark.table(table_name)
        return True
    except:
        return False

def drop_table_if_exists(spark, table_name):
    """删除表（如果存在）"""
    if table_exists(spark, table_name):
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")
        print(f"已删除临时表 {table_name}")

def refresh_related_tables(spark):
    """刷新所有查询用到的表，确保读取最新数据"""
    # 整理所有用到的表清单
    related_tables = [
        "dwd.dwd_gis_progress_report_f_1h",
        "dwd.dwd_gis_dispatch_table_f_1h",
        "dwd.dwd_gis_fleet_info_i_1mon",
        "dwd.dwd_gis_vehicle_model_info_i_1mon",
        "dwd.dwd_gis_fn_nw_company_info_i_1mon",
        "dwd.dwd_gis_parking_lot_queue_i_1h",
        "dwd.dwd_gis_gross_weight_f_1h",
        "dwd.dwd_gis_dispatch_plan_info_i_1h"
    ]
    
    print("\n===== 开始刷新相关表 =====")
    for table in related_tables:
        try:
            # 执行Hive的REFRESH TABLE命令，刷新表元数据和数据缓存
            spark.sql(f"REFRESH TABLE {table}")
            print(f"✅ 成功刷新表：{table}")
        except Exception as e:
            # 表不存在或刷新失败时给出提示，但不中断程序
            print(f"⚠️  刷新表 {table} 失败：{str(e)}（非关键错误，继续执行）")
    print("===== 表刷新操作完成 =====\n")

def create_target_table(spark, table_name):
    """
    显式创建非分区目标表，强制指定字段类型（核心：tonnage/estimated_weight为DOUBLE）
    适配Hive表结构，存储格式为parquet（主流列式存储）
    """
    create_sql = f"""
CREATE TABLE IF NOT EXISTS {table_name} (
    factory_no STRING COMMENT '工厂编号',
    factorycode STRING COMMENT '工厂编码',
    season STRING COMMENT '季节',
    inouttype STRING COMMENT '进出类型编码',
    inouttypename STRING COMMENT '进出类型名称',
    cxdm STRING COMMENT '车型代码',
    cxmc STRING COMMENT '车型名称',
    vehicle_type_code STRING COMMENT '车辆类型编码',
    vehicle_type STRING COMMENT '车辆类型',
    datetarget STRING COMMENT '目标日期',
    vehicle_num STRING COMMENT '车辆数量',
    tonnage DOUBLE COMMENT '单车载重（吨）',  -- 强制DOUBLE类型
    estimated_weight DOUBLE COMMENT '预估总重量（吨）',  -- 强制DOUBLE类型
    create_time STRING COMMENT '创建时间',
    update_time STRING COMMENT '更新时间',
    source_flg STRING COMMENT '数据来源标识',
    dw_update_time STRING COMMENT '数仓更新时间',
    dt STRING COMMENT '数据日期（非分区字段）'  -- 保留dt字段但不作为分区
)
STORED AS PARQUET  -- 列式存储，适配大数据场景
TBLPROPERTIES (
    'parquet.compression'='snappy',  -- 压缩格式
    'skip.header.line.count'='0',
    'transient_lastDdlTime'='{int(datetime.now().timestamp())}'
)
    """
    spark.sql(create_sql)
    print(f"✅ 成功创建/验证非分区目标表 {table_name}，字段类型已强制定义（tonnage/estimated_weight为DOUBLE）")

# ===================== 4. 主逻辑 =====================
print("===== 开始启动SparkSession =====")
# 创建SparkSession（移除无效配置，适配Spark 3.0+）
spark = SparkSession.builder \
    .appName("VehicleQueuingData") \
    .enableHiveSupport() \
    .getOrCreate()
print("===== SparkSession 启动成功 =====")

# 表名定义
target_table = "app.app_gis_vehicle_queues_f_1d"
history_table = "app.app_dm_dsr_vehicle_queuing_f_1y"
# 生成唯一临时表名和外部表路径（避免冲突）
temp_table_name = f"app.tmp_vehicle_{current_date_str.replace('-', '')}_{str(uuid.uuid4())[:8]}"
temp_table_path = f"/user/hive/warehouse/app.db/{temp_table_name}"  # 适配Hive默认路径

# 检查目标表是否存在
target_table_exists = table_exists(spark, target_table)

if not target_table_exists:
    # 目标表不存在：第一步先显式创建非分区表，再初始化数据
    print(f"\n===== 目标表 {target_table} 不存在，先显式创建非分区表结构 =====")
    create_target_table(spark, target_table)
    
    # 第二步：尝试从历史表初始化数据
    print(f"\n===== 尝试从历史表 {history_table} 初始化目标表数据 =====")
    history_table_exists = table_exists(spark, history_table)
    
    if history_table_exists:
        history_df = spark.table(history_table)
        # 过滤历史表字段，只保留目标表需要的列（避免多余字段导致类型冲突）
        target_cols = [
            "factory_no", "factorycode", "season", "inouttype", "inouttypename",
            "cxdm", "cxmc", "vehicle_type_code", "vehicle_type", "datetarget",
            "vehicle_num", "tonnage", "estimated_weight", "create_time",
            "update_time", "source_flg", "dw_update_time", "dt"
        ]
        
        # 补充历史表缺失的列（用默认值填充）
        for col_name in target_cols:
            if col_name not in history_df.columns:
                print(f"⚠️  历史表缺失列 {col_name}，用默认值填充")
                if col_name in ["tonnage", "estimated_weight"]:
                    # 数值字段默认0.0，显式指定Double类型
                    history_df = history_df.withColumn(col_name, lit(0.0).cast(DoubleType()))
                else:
                    # 字符串字段默认空串
                    history_df = history_df.withColumn(col_name, lit("").cast(StringType()))
        
        # 强制转换历史表的tonnage/estimated_weight为Double（防止历史表类型不符）
        history_df = history_df \
            .withColumn("tonnage", col("tonnage").cast(DoubleType())) \
            .withColumn("estimated_weight", col("estimated_weight").cast(DoubleType()))
        
        # 只保留目标列，写入非分区表
        history_df = history_df.select(target_cols)
        history_count = history_df.count()
        
        # 写入非分区表（无partitionBy）
        history_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"✅ 成功从历史数据表 {history_table} 初始化目标表，共写入 {history_count} 条数据")
    else:
        print(f"⚠️  历史数据表 {history_table} 不存在，目标表已创建空表（字段类型已保证）")
else:
    # 目标表存在：增量处理（修正后逻辑）
    print(f"\n===== 开始处理 {current_date_str} 日期增量数据 =====")

    # 步骤1：查询目标表对应日期的存量数据条数（核心修改：dt → datetarget）
    stock_df = spark.table(target_table).filter(col("datetarget") == current_date_str)
    stock_count = stock_df.count()
    print(f"目标表 {target_table} 中 {current_date_str} 日期的存量数据条数：{stock_count}")

    # 步骤2：删除对应日期数据（核心修改：dt → datetarget）
    if stock_count > 0:
        # 第一步：清理残留的临时表（防止路径冲突）
        drop_table_if_exists(spark, temp_table_name)
        
        # 第二步：将目标表中「非当前日期」的数据写入临时表（修改dt为datetarget）
        print(f"创建临时表 {temp_table_name} 存储非当前日期数据")
        remain_df = spark.table(target_table).filter(col("datetarget") != current_date_str)
        remain_df.write \
            .mode("overwrite") \
            .option("path", temp_table_path) \
            .saveAsTable(temp_table_name)
        
        # 第三步：清空原目标表，将临时表数据写回（实现删除当前日期数据）
        spark.sql(f"TRUNCATE TABLE {target_table}")  # 清空非分区表
        remain_df = spark.table(temp_table_name)
        remain_df.write.mode("overwrite").saveAsTable(target_table)
        
        # 第四步：验证删除结果（修改dt为datetarget）
        after_delete_count = spark.table(target_table).filter(col("datetarget") == current_date_str).count()
        delete_count = stock_count - after_delete_count
        print(f"成功删除 {current_date_str} 日期的 {delete_count} 条数据（删除后剩余 {after_delete_count} 条）")
        
        # 第五步：清理临时表（元数据+存储）
        drop_table_if_exists(spark, temp_table_name)
        try:
            os.system(f"hadoop fs -rm -r {temp_table_path}")
            print(f"已清理临时表存储路径 {temp_table_path}")
        except:
            print(f"临时表路径 {temp_table_path} 清理失败（非必要，不影响核心逻辑）")
    else:
        print(f"目标表中无 {current_date_str} 日期数据，无需删除")
        delete_count = 0
    
    # ===================== 关键修改：执行查询前刷新所有相关表 =====================
    refresh_related_tables(spark)
    
    # 步骤3：调用方法生成SQL并执行查询
    print(f"\n开始查询 {current_date_str} 日期的车辆数据")
    # 返途中车辆
    print("执行第一部分查询：返途中车辆")
    df1 = spark.sql(get_sql_part1(current_date_str))
    # 厂外排队车辆
    print("执行第二部分查询：厂外排队车辆")
    df2 = spark.sql(get_sql_part2(current_date_str))
    # 厂内车辆
    print("执行第三部分查询：厂内车辆")
    df3 = spark.sql(get_sql_part3(current_date_str))
    
    # 步骤4：统一列结构并合并
    all_columns = set(df1.columns) | set(df2.columns) | set(df3.columns)
    for col_name in all_columns:
        if col_name not in df1.columns:
            df1 = df1.withColumn(col_name, lit(None).cast(StringType()))
        if col_name not in df2.columns:
            df2 = df2.withColumn(col_name, lit(None).cast(StringType()))
        if col_name not in df3.columns:
            df3 = df3.withColumn(col_name, lit(None).cast(StringType()))
    current_df = df1.unionByName(df2).unionByName(df3)
    
    # 步骤5：Schema对齐（和目标表匹配）
    print("\n开始对齐数据schema与目标表")
    current_df = current_df \
        .withColumnRenamed("factoryCode", "factorycode") \
        .withColumnRenamed("factory_No", "factory_no") \
        .withColumnRenamed("dateTarget", "datetarget") \
        .withColumnRenamed("Estimated_weight", "estimated_weight")
    
    # 类型转换（双重保障）
    current_df = current_df \
        .withColumn("vehicle_num", col("vehicle_num").cast(StringType())) \
        .withColumn("tonnage", col("tonnage").cast(DoubleType())) \
        .withColumn("estimated_weight", col("estimated_weight").cast(DoubleType())) \
        .withColumn("create_time", date_format(col("create_time"), "yyyy-MM-dd HH:mm:ss").cast(StringType()))
    
    # 补充缺失列
    current_df = current_df \
        .withColumn("update_time", col("create_time")) \
        .withColumn("source_flg", lit("gis")) \
        .withColumn("dw_update_time", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss").cast(StringType())) \
        .withColumn("dt", lit(current_date_str))  # 保留dt字段（非分区）
    
    # 调整列顺序（和目标表Schema完全一致）
    target_columns = [
        "factory_no", "factorycode", "season", "inouttype", "inouttypename",
        "cxdm", "cxmc", "vehicle_type_code", "vehicle_type", "datetarget",
        "vehicle_num", "tonnage", "estimated_weight", "create_time",
        "update_time", "source_flg", "dw_update_time", "dt"
    ]
    current_df = current_df.select(target_columns)
    
    # 步骤6：统计写入条数并追加写入目标表（非分区表append模式）
    write_count = current_df.count()
    current_df.write.mode("append").saveAsTable(target_table)
    
    # 步骤7：打印最终统计结果
    print(f"\n===== {current_date_str} 日期数据处理完成 =====")
    if stock_count == 0:
        print(f"目标表中无 {current_date_str} 日期数据，本次新增 {write_count} 条数据")
    else:
        print(f"目标表中 {current_date_str} 日期原有 {stock_count} 条数据，删除 {delete_count} 条后，重新写入 {write_count} 条数据")

# 最终验证：打印目标表的字段类型（可选，用于调试）
print(f"\n===== 验证目标表 {target_table} 的字段类型 =====")
target_schema = spark.table(target_table).schema
for field in target_schema.fields:
    if field.name in ["tonnage", "estimated_weight"]:
        print(f"✅ {field.name}: {field.dataType}（预期：DoubleType）")

# 最终清理：确保临时表被删除（防止残留）
drop_table_if_exists(spark, temp_table_name)

# 停止SparkSession
print("\n===== 开始关闭SparkSession =====")
spark.stop()
print("===== SparkSession 关闭成功 =====")

===== 开始启动SparkSession =====
===== SparkSession 启动成功 =====

===== 开始处理 2026-03-17 日期增量数据 =====
目标表 app.app_gis_vehicle_queues_f_1d 中 2026-03-17 日期的存量数据条数：16
创建临时表 app.tmp_vehicle_20260317_0ec30ea6 存储非当前日期数据
成功删除 2026-03-17 日期的 16 条数据（删除后剩余 0 条）
已删除临时表 app.tmp_vehicle_20260317_0ec30ea6
已清理临时表存储路径 /user/hive/warehouse/app.db/app.tmp_vehicle_20260317_0ec30ea6

===== 开始刷新相关表 =====
✅ 成功刷新表：dwd.dwd_gis_progress_report_f_1h
⚠️  刷新表 dwd.dwd_gis_dispatch_table_f_1h 失败：Table or view not found: dwd.dwd_gis_dispatch_table_f_1h; line 1 pos 14;
'RefreshTable
+- 'UnresolvedTableOrView [dwd, dwd_gis_dispatch_table_f_1h], REFRESH TABLE, true
（非关键错误，继续执行）
✅ 成功刷新表：dwd.dwd_gis_fleet_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_vehicle_model_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_fn_nw_company_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_parking_lot_queue_i_1h
✅ 成功刷新表：dwd.dwd_gis_gross_weight_f_1h
✅ 成功刷新表：dwd.dwd_gis_dispatch_plan_info_i_1h
===== 表刷新操作完成 =====


开始查询 2026-03-17 日期的车辆数据
执行第一部分查询：返途中车辆


AnalysisException: Table or view not found: dwd.dwd_gis_dispatch_table_f_1h; line 36 pos 11;
'Aggregate [CASE WHEN (('j8.factory = 6011) AND ('j10.xh = 2)) THEN FNR WHEN (('j8.factory = 6041) AND ('j10.xh = 2)) THEN NMR ELSE 'comp.pyjiancheng END, CASE WHEN (('j8.factory = 6011) AND ('j10.xh = 2)) THEN 6071 WHEN (('j8.factory = 6041) AND ('j10.xh = 2)) THEN 60412 ELSE 'j8.factory END, 'CONCAT(20, 'j8.season), 'z31.fl, 'z31.cmc, 'z32.cllxdm, 'z32.cllx], [CASE WHEN (('j8.factory = 6011) AND ('j10.xh = 2)) THEN FNR WHEN (('j8.factory = 6041) AND ('j10.xh = 2)) THEN NMR ELSE 'comp.pyjiancheng END AS factoryCode#1389, CASE WHEN (('j8.factory = 6011) AND ('j10.xh = 2)) THEN 6071 WHEN (('j8.factory = 6041) AND ('j10.xh = 2)) THEN 60412 ELSE 'j8.factory END AS factory_No#1390, 'CONCAT(20, 'j8.season) AS season#1391, 1 AS inouttype#1392, 返途中to_factory AS inouttypename#1393, 'z31.fl AS cxdm#1394, 'z31.cmc AS cxmc#1395, 'z32.cllxdm AS vehicle_type_code#1396, 'z32.cllx AS vehicle_type#1397, 2026-0